# LLM Comparison for Resume-Job Matching

Our main scoring model is Sentence-BERT, but the proposal feedback asked us to also try a current LLM. So in this notebook we use FLAN-T5-base.

We do two things:
1. Ask the LLM to label each original resume against the job in its domain (Strong, Partial, or Weak match).
2. Ask the LLM the same question for the counterfactual versions of each resume, so we can check whether the LLM's label flips when only the name, pronouns, or university change.

Note: In our first run we used beam search with the options listed in a fixed order. The model returned the same label for every input. To avoid that, we now sample with a temperature and shuffle the order of the options in the prompt each time.

In [ ]:
!pip install transformers torch pandas

In [ ]:
import os
import random
import pandas as pd
import torch

random.seed(42)
torch.manual_seed(42)

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded.")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs:", jobs.shape)
print("Resumes:", resumes.shape)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("FLAN-T5-base loaded.")

In [ ]:
OPTIONS = ["Strong match", "Partial match", "Weak match"]

def ask_llm(resume_text, job_text):
    # Shuffle the options so the model is not biased by the order in the prompt.
    options = OPTIONS.copy()
    random.shuffle(options)
    options_str = ", ".join(options)

    prompt = (
        "Read the job description and the resume. "
        f"Pick one of: {options_str}. "
        "Answer with just the label.\n\n"
        f"Job: {job_text[:900]}\n\n"
        f"Resume: {resume_text[:900]}\n\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Map free-text answer back to one of our three labels.
    text = raw.lower()
    if "strong" in text:
        return "Strong match"
    if "partial" in text:
        return "Partial match"
    if "weak" in text:
        return "Weak match"
    return raw

In [ ]:
def make_job_text(row):
    return f"{row['title']} ({row['domain']}) at {row['company_name']}. {row['job_description']}"

jobs["job_text"] = jobs.apply(make_job_text, axis=1)

In [ ]:
# Run the LLM on every resume variant, matched to the job in the same domain.
# This way we can see whether the LLM's label changes when only a demographic
# signal is changed.
results = []
for _, resume_row in resumes.iterrows():
    matching_jobs = jobs[jobs["domain"] == resume_row["domain"]]
    if len(matching_jobs) == 0:
        continue
    job_row = matching_jobs.iloc[0]
    label = ask_llm(resume_row["resume_text"], job_row["job_text"])
    results.append({
        "resume_id": resume_row["resume_id"],
        "version": resume_row["version"],
        "changed_signal": resume_row["changed_signal"],
        "resume_domain": resume_row["domain"],
        "job_id": job_row["job_id"],
        "job_title": job_row["title"],
        "llm_match_decision": label,
    })

llm_df = pd.DataFrame(results)
print("Total LLM calls:", len(llm_df))
display(llm_df.head(10))
print("Label counts:")
print(llm_df["llm_match_decision"].value_counts())

In [ ]:
# Did the LLM change its decision when only a demographic signal changed?
original_labels = (
    llm_df[llm_df["version"] == "original"]
    [["resume_id", "llm_match_decision"]]
    .rename(columns={"llm_match_decision": "original_label"})
)

changed = llm_df[llm_df["version"] != "original"].merge(
    original_labels, on="resume_id", how="left"
)
changed["label_changed"] = changed["llm_match_decision"] != changed["original_label"]

flip_summary = (
    changed.groupby("changed_signal")["label_changed"]
    .mean()
    .reset_index()
    .rename(columns={"label_changed": "fraction_of_label_flips"})
)
display(flip_summary)

In [ ]:
llm_df.to_csv("results/llm_match_decisions.csv", index=False)
changed.to_csv("results/llm_counterfactual_comparison.csv", index=False)
flip_summary.to_csv("results/llm_flip_summary.csv", index=False)
print("Saved LLM result files.")

In [ ]:
from google.colab import files
files.download("results/llm_match_decisions.csv")
files.download("results/llm_counterfactual_comparison.csv")
files.download("results/llm_flip_summary.csv")